# Learning Objectives

In this notebook, you will craft sophisticated ETL jobs that interface with a variety of common data sources, such as 
- REST APIs (HTTP endpoints)
- RDBMS
- Hive tables (managed tables)
- Various file formats (csv, json, parquet, etc.)

d

# Interview Questions

As you progress through the practice, attempt to answer the following questions:

## Columnar File
- What is a columnar file format and what advantages does it offer?
- Why is Parquet frequently used with Spark and how does it function?
- How do you read/write data from/to a Parquet file using a DataFrame?

## Partitions
- How do you save data to a file system by partitions? (Hint: Provide the code)
- How and why can partitions reduce query execution time? (Hint: Give an example)

## JDBC and RDBMS
- How do you load data from an RDBMS into Spark? (Hint: Discuss the steps and JDBC)

## REST API and HTTP Requests
- How can Spark be used to fetch data from a REST API? (Hint: Discuss making API requests)

## ETL Job One: Parquet file
### Extract
Extract data from the managed tables (e.g. `bookings_csv`, `members_csv`, and `facilities_csv`)

### Transform
Data transformation requirements https://pgexercises.com/questions/aggregates/fachoursbymonth.html

### Load
Load data into a parquet file

### What is Parquet? 

Columnar files are an important technique for optimizing Spark queries. Additionally, they are often tested in interviews.
- https://www.youtube.com/watch?v=KLFadWdomyI
- https://www.databricks.com/glossary/what-is-parquet

In [0]:
from pyspark.sql.functions import to_date,sum,concat,lit

In [0]:
%sql
USE jarvis.bronze

In [0]:
bookings_df = spark.sql('SELECT * FROM bookings')
members_df = spark.sql('SELECT * FROM members')
facilities_df = spark.sql('SELECT * FROM facilities')

write_location = "/Volumes/jarvis/silver/data"

In [0]:
'''
SELECT facid, SUM(slots) as "Total Slots" FROM cd.bookings
  WHERE starttime >= '2012-09-01' AND starttime < '2012-10-01'
  GROUP BY facid
  ORDER BY SUM(slots)
'''
fachoursbymonth = bookings_df.join(facilities_df, 'facid', 'inner').filter(to_date(bookings_df.starttime).between('2012-09-01', '2012-09-30')).groupBy('facid').agg(sum('slots').alias('Total Slots')).orderBy("Total Slots")
file_name = "fachoursbymonth.parquet"
fachoursbymonth.write.mode('overwrite').parquet(f"{write_location}/{file_name}")


## ETL Job Two: Partitions

### Extract
Extract data from the managed tables (e.g. `bookings_csv`, `members_csv`, and `facilities_csv`)

### Transform
Transform the data https://pgexercises.com/questions/joins/threejoin.html

### Load
Partition the result data by facility column and then save to `threejoin_delta` managed table. Additionally, they are often tested in interviews.

hint: https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrameWriter.partitionBy.html

What are paritions? 

Partitions are an important technique to optimize Spark queries
- https://www.youtube.com/watch?v=hvF7tY2-L3U&t=268s

In [0]:
# Write your solution here

three_join = bookings_df.join(members_df, 'memid', 'inner').join(facilities_df, 'facid', 'inner').filter(facilities_df.name.like('Tennis Court%')).select(
    concat(members_df.firstname, lit(' '), members_df.surname).alias('member'),
    facilities_df.name.alias('facility')).dropDuplicates().orderBy('member', 'facility')

three_joinTable = "threejoin_delta"

three_join.write.mode('overwrite').partitionBy('facility').saveAsTable(three_joinTable)

## ETL Job Three: HTTP Requests

### Extract
Extract daily stock price data price from the following companies, Google, Apple, Microsoft, and Tesla. 

Data Source
- API: https://rapidapi.com/alphavantage/api/alpha-vantage
- Endpoint: GET `TIME_SERIES_DAILY`

Sample HTTP request

```
curl --request GET \
	--url 'https://alpha-vantage.p.rapidapi.com/query?function=TIME_SERIES_DAILY&symbol=TSLA&outputsize=compact&datatype=json' \
	--header 'X-RapidAPI-Host: alpha-vantage.p.rapidapi.com' \
	--header 'X-RapidAPI-Key: [YOUR_KEY]'

```

Sample Python HTTP request

```
import requests

url = "https://alpha-vantage.p.rapidapi.com/query"

querystring = {
    "function":"TIME_SERIES_DAILY",
    "symbol":"IBM",
    "datatype":"json",
    "outputsize":"compact"
}

headers = {
    "X-RapidAPI-Host": "alpha-vantage.p.rapidapi.com",
    "X-RapidAPI-Key": "[YOUR_KEY]"
}

response = requests.get(url, headers=headers, params=querystring)

data = response.json()

# Now 'data' contains the daily time series data for "IBM"
```

### Transform
Find **weekly** max closing price for each company.

hints: 
  - Use a `for-loop` to get stock data for each company
  - Use the spark `union` operation to concat all data into one DF
  - create a new `week` column from the data column
  - use `group by` to calcualte max closing price

### Load
- Partition `DF` by company
- Load the DF in to a managed table called, `max_closing_price_weekly`

In [0]:

import requests
from functools import reduce
from pyspark.sql import DataFrame, functions as F

url = "https://alpha-vantage.p.rapidapi.com/query"
headers = {
    'x-rapidapi-key': "0dd0550f3cmshc01bcb41ee0fcbap1262e6jsn3dc6b85248d5",
    'x-rapidapi-host': "alpha-vantage.p.rapidapi.com",
}


In [0]:
def getStockData(company):
    querystring = {
        "function": "TIME_SERIES_DAILY",
        "symbol": f"{company}",
        "datatype": "json",
        "outputsize": "compact"
    }
    
    #ensure that the request is successful and handle any potential errors
    try:
        response = requests.get(url, headers=headers, params=querystring, timeout=5)
        response.raise_for_status()
        payload = response.json()
        data = payload["Time Series (Daily)"]
    #generic fails 
    except requests.exceptions.RequestException as e:
        print(f"Request failed for {company}: {e}")
        return None
    #handle missings
    except KeyError:
        print(f"Unexpected response for {company}: {payload}")
        return None
    
    # create a DataFrame from the time series data
    df = spark.createDataFrame([{"timeseries": data}])
    df = df.select(F.explode("timeseries").alias("date", "metrics"))

    company_df = (df
        .select(
            "date",
            F.col("metrics")["1. open"].cast("double").alias("open"),
            F.col("metrics")["2. high"].cast("double").alias("high"),
            F.col("metrics")["3. low"].cast("double").alias("low"),
            F.col("metrics")["4. close"].cast("double").alias("close"),
            F.col("metrics")["5. volume"].cast("long").alias("volume"),
        )
        .withColumn("symbol", F.lit(f"{company}"))
    )
       
       
    # date comes back as a string, cast it to a real date
    company_df = company_df.withColumn("date", F.col("date").cast("date"))
    return company_df


companies = ['GOOGL', 'AAPL', 'MSFT', 'TSLA']

dfs = [df for df in [getStockData(company) for company in companies]]
stock_price_df = reduce(DataFrame.union, dfs)
display(stock_price_df.head(5))

date,open,high,low,close,volume,symbol
2026-07-09,354.31,359.65,351.08,358.89,24729093,GOOGL
2026-07-08,364.76,367.84,358.02,361.92,22094769,GOOGL
2026-07-07,369.07,373.16,365.5,367.03,24045581,GOOGL
2026-07-06,361.55,367.93,357.3805,366.46,26915808,GOOGL
2026-07-02,359.48,364.205,353.42,359.91,25999346,GOOGL


In [0]:
#create the week column
stock_price_with_week_df = stock_price_df.withColumn(
    "week", F.date_trunc("week", F.col("date"))
)

weekly_max_close_df = (stock_price_with_week_df
    .groupBy("symbol", "week")
    .agg(F.max("close").alias("max_close"))
    .orderBy("symbol", "week")
)

display(weekly_max_close_df.head(5))
weekly_max_close_df.write.mode('overwrite').option("overwriteSchema", "true").partitionBy('symbol').saveAsTable("max_closing_price_weekly")


symbol,week,max_close
AAPL,2026-02-09T00:00:00.000Z,255.78
AAPL,2026-02-16T00:00:00.000Z,264.58
AAPL,2026-02-23T00:00:00.000Z,274.23
AAPL,2026-03-02T00:00:00.000Z,264.72
AAPL,2026-03-09T00:00:00.000Z,260.83


## ETL Job Four: RDBMS


### Extract
Extract RNA data from a public PostgreSQL database.

- https://rnacentral.org/help/public-database
- Extract 100 RNA records from the `rna` table (hint: use `limit` in your sql)
- hint: use `spark.read.jdbc` https://docs.databricks.com/external-data/jdbc.html

### Transform
We want to load the data as it so there is no transformation required.


### Load
Load the DF in to a managed table called, `rna_100_records`

In [0]:
table_name = "rna"
db_URL = "hh-pgsql-public.ebi.ac.uk"
port = 5432
db_name = "pfmegrnargs"
username = "reader"
password = "NWDMCE5xdipIjRrp"

In [0]:
rna_df_100 = (spark.read
  .format("jdbc")
  .option("url", f"jdbc:postgresql://{db_URL}:{port}/{db_name}")
  .option("dbtable", f"(SELECT * FROM {table_name} LIMIT 10) AS t")
  .option("user", username)
  .option("password", password)
  .load()
)

rna_df_100.write.mode("overwrite").saveAsTable("rna_100_records")

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-8490470533413589>, line 10
      1 rna_df_100 = (spark.read
      2   .format("jdbc")
      3   .option("url", f"jdbc:postgresql://{db_URL}:{port}/{db_name}")
   (...)
      7   .load()
      8 )
---> 10 rna_df_100.write.mode("overwrite").saveAsTable("rna_100_records")

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/readwriter.py:737, in DataFrameWriter.saveAsTable(self, name, format, mode, partitionBy, **options)
    735 self._write.table_name = name
    736 self._write.table_save_method = "save_as_table"
--> 737 _, _, ei = self._spark.client.execute_command(
    738     self._write.command(self._spark.client), self._write.observations
    739 )
    740 self._callback(ei)

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/client/core.py:1538, in SparkConnectClient.execute_c